# 05 Segmentación de Clientes con Modelo RFM

En este ejercicio vamos a transformar datos transaccionales crudos en segmentos de clientes accionables.

**Pasos:**
1. Generación, Carga y Exploración de Datos
2. Cálculo de Métricas RFM
3. Scoring
4. Segmentación Estratégica para la Toma de Decisiones
5. Visualización    

## Paso 1: Generación, Carga y Exploración de Datos:


In [1]:
import pandas as pd
import numpy as np
import datetime

In [2]:
np.random.seed(42)
n_transacciones = 1000
n_clientes = 50

In [3]:
fechas = pd.date_range(start='2025-07-01', end='2026-06-30', periods=n_transacciones)
fechas = np.random.choice(fechas, n_transacciones) # Desordenar fechas
cliente_ids = np.random.randint(1000, 1000 + n_clientes, n_transacciones)
cantidades = np.random.randint(1, 10, n_transacciones)
precios = np.round(np.random.uniform(10, 100, n_transacciones), 2)

In [4]:
df = pd.DataFrame({
    'ID_Transaccion': [f'TRX-{i}' for i in range(n_transacciones)],
    'ID_Cliente': cliente_ids,
    'Fecha': fechas,
    'Cantidad': cantidades,
    'Precio_Unitario': precios
})

df['Total'] = df['Cantidad'] * df['Precio_Unitario']

In [5]:
df

,ID_Transaccion,ID_Cliente,Fecha,Cantidad,Precio_Unitario,Total
0,TRX-0,1044,2025-08-07 03:57:50.270270,5,30.28,151.40
1,TRX-1,1004,2025-12-06 11:57:50.270270,9,32.59,293.31
2,TRX-2,1032,2026-05-10 08:28:49.729729,3,86.56,259.68
3,TRX-3,1000,2025-10-07 09:04:51.891891,7,60.51,423.57
4,TRX-4,1017,2025-08-08 14:56:34.594594,5,57.11,285.55
...,...,...,...,...,...,...
995,TRX-995,1041,2025-07-04 06:42:09.729729,9,19.12,172.08
996,TRX-996,1035,2026-04-26 20:55:29.729729,9,35.01,315.09
997,TRX-997,1026,2026-04-17 09:33:41.621621,1,34.87,34.87
998,TRX-998,1037,2025-09-26 19:29:00.540540,2,48.88,97.76


## Paso 2: Cáculo de Métricas RFM

### 2.1: Recencia
Calcular hace cuántos días fue la última compra de cada uno de los clientes

In [6]:
df['Fecha'].max()

Timestamp('2026-06-29 15:15:18.918918')

In [7]:
df_rfm = pd.DataFrame(df.groupby('ID_Cliente')['Fecha'].max())
df_rfm

,Fecha
ID_Cliente,
1000,2026-06-19 01:39:27.567567
1001,2026-06-28 21:45:56.756756
1002,2026-06-28 21:45:56.756756
1003,2026-06-21 06:07:34.054054
1004,2026-06-23 10:35:40.540540
1005,2026-06-28 13:01:15.675675
1006,2026-05-30 00:41:48.108108
1007,2026-06-29 15:15:18.918918
1008,2026-06-20 12:38:11.891891


In [ ]:
# Aquí deben hacer el código para las demás métricas: Frecuencia y Monetario

## Parte 3: Scoring

Usa la función pd.qcut para dividir los datos en terciles:

Para Frecuencia y Monetario:

* Etiqueta 3 = Valores más altos (Mejor).
* Etiqueta 2 = Valores medios.
* Etiqueta 1 = Valores más bajos (Peor).

Para Recencia la lógica es inversa: Un valor bajo de recencia es mejor que un valor alto.

* Etiqueta 3 = Valores más bajos de días (Compró hace poco).
* Etiqueta 2 = Valores medios.
* Etiqueta 1 = Valores más altos de días (Hace mucho no compra).


## Parte 4: Segmentación Estratégica

Crea una columna llamada Segmento basada en las siguientes reglas lógicas:

| Segmento  | Regla                 | Acción de Negocio                                             |
| --------- | --------------------- | ------------------------------------------------------------- |
| Campeones | R=3, F=3, M=3         | No molestar.                                                  |
| Leales    | F=3 (cualquier R o M) | Intentar subir su ticket promedio (*up-selling*).             |
| En riesgo | R=1, F=3 y/o M=3      | **Urgente:** enviar una promoción agresiva para reactivarlos. |
| Perdidos  | R=1, F=1, M=1         | No invertir presupuesto de marketing.                         |
| Nuevos    | R=3, F=1              | Programa de bienvenida (*onboarding*).                        |
